<a href="https://colab.research.google.com/github/Mohammed-Taha20/OCR/blob/main/DEPI_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# prepare the Data

In [ ]:
import os
import random
from shutil import copyfile

# Set your paths
base_path = "/content/drive/MyDrive/DEPI_Project_Dataset/Arabic_Data"
img_folder = os.path.join(base_path, "img")
ann_folder = os.path.join(base_path, "ann")
output_folder = "/content/MIX_Data_Sample"  # Folder to save the sample

# Create output folders if they don't exist
os.makedirs(os.path.join(output_folder, "img"), exist_ok=True)
os.makedirs(os.path.join(output_folder, "ann"), exist_ok=True)

# Get list of all files (assuming matching names between img and ann)
img_files = [f for f in os.listdir(img_folder) if f.endswith(('.jpg', '.jpeg', '.png'))]
ann_files = [f.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt') for f in img_files]

# Verify we have matching pairs
assert len(img_files) == len(ann_files), "Mismatch between image and annotation files"

# Randomly select 100 pairs
selected_pairs = random.sample(list(zip(img_files, ann_files)), 200)

# Copy selected files to output folder
for img_file, ann_file in selected_pairs:
    # Copy image
    src_img = os.path.join(img_folder, img_file)
    dst_img = os.path.join(output_folder, "img", img_file)
    copyfile(src_img, dst_img)

    # Copy annotation
    src_ann = os.path.join(ann_folder, ann_file)
    dst_ann = os.path.join(output_folder, "ann", ann_file)
    copyfile(src_ann, dst_ann)

print(f"Successfully copied 100 random samples to {output_folder}")

In [ ]:
import shutil

source_folder = "/content/MIX_Data_Sample"
destination_folder = "/content/drive/MyDrive/DEPI_Project_Dataset"

shutil.copytree(source_folder, os.path.join(destination_folder, "MIX_Data_Sample"), dirs_exist_ok=True)
print(f"Successfully copied the '{source_folder}' folder to '{destination_folder}'")

Successfully copied the '/content/MIX_Data_Sample' folder to '/content/drive/MyDrive/DEPI_Project_Dataset'


# EZOCR

In [ ]:
!pip install easyocr opencv-python matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.8/422.8 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
import easyocr

# Initialize the reader (you can include both Arabic and English if your data has both)
reader = easyocr.Reader(['en', 'ar'])  # Add 'ar' if you're working with Arabic


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
import os
import json
from glob import glob
from difflib import SequenceMatcher
import easyocr
import cv2
import matplotlib.pyplot as plt


## testing

In [ ]:

# Path to your image
image_path = '/content/drive/MyDrive/DEPI_Project_Dataset/MIX_Data_Sample/img/06ddca9d-2259-4973-bc2a-beb312330c90_date.jpg'  # Replace with your actual image path

# Load the image using OpenCV
image = cv2.imread(image_path)
if image is None:
    raise FileNotFoundError(f"Image not found at {image_path}")

# Initialize EasyOCR Reader with Arabic (and optionally English)
reader = easyocr.Reader(['ar', 'en'])  # You can adjust languages here

# Perform OCR
results = reader.readtext(image,paragraph = True,detail=0)
results
# Draw results on image
for (bbox, text) in results:
    # Get bounding box points
    top_left = tuple(map(int, bbox[0]))
    bottom_right = tuple(map(int, bbox[2]))

    # Draw rectangle
    cv2.rectangle(image, top_left, bottom_right, (0, 255, 0), 2)

    # Put detected text above the box
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(image, text, (top_left[0], top_left[1] - 10), font, 0.7, (0, 0, 255), 2, cv2.LINE_AA)
    print(text)
# Convert BGR to RGB for display
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Show the image with text
plt.figure(figsize=(10, 10))
plt.imshow(image_rgb)
plt.axis('off')
plt.title('Detected Text with EasyOCR')
plt.show()


['29.7', '4.95', 'حليب كامل الدىم 200مل']

## predicting func

In [ ]:
def predict_text(image_path):
    # Load the image using OpenCV
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Image not found at {image_path}")

    # Initialize EasyOCR Reader with Arabic (and optionally English)
    reader = easyocr.Reader(['ar', 'en'])  # You can adjust languages here

    # Perform OCR
    results = reader.readtext(image,paragraph = True,detail=0)
    results = ' '.join(results)
    return results

##similarity

In [ ]:
# Helper function for text similarity
def average_similarity(ground_truths,perdicted):

    def string_similarity(a, b):
        return SequenceMatcher(None, a, b).ratio()

    # Calculate average similarity
    if perdicted:
        similarities = [string_similarity(pred, gt) for pred, gt in zip(perdicted, ground_truths)]
        average_similarity = sum(similarities) / len(similarities)
        print(f"\n✅ Average OCR similarity: {average_similarity:.2f}")
        return average_similarity
    else:
        print("\n⚠️ No valid predictions made.")
        return -1.0


##EZOCR Dict

In [ ]:
base_path = '/content/drive/MyDrive/DEPI_Project_Dataset/MIX_Data_Sample'


In [ ]:
def get_label(image_path):
    try:
        ann_path = image_path.replace('.jpg','.txt').replace('img','ann')
        with open(ann_path, 'r') as f:
            lines = f.readline()
            lines=eval(lines)
            result = ' '.join(lines)
            return result
    except FileNotFoundError:
        print(f"File not found: {ann_path}")



In [ ]:
predictions = []
data_dict={'image path':[] , 'labled':[],'predicted_ezOCR' :[],'similarities_ezOCR':[]}

for img in glob(os.path.join(base_path, 'img', '*')):

    results_text=predict_text(img)
    data_dict['image path'].append(img)
    data_dict['predicted_ezOCR'].append(results_text)
    label=get_label(img)
    print(f'predicted : {results_text}')
    print(f'label : {label}')
    data_dict['labled'].append(label)
    data_dict['similarities_ezOCR'].append(average_similarity(label,results_text))
    predictions.append(results_text)



predicted : SALE
label : SALE

✅ Average OCR similarity: 1.00
predicted : Invoice Date: 30-10-2021
label : Invoice Date: 30-10-2021

✅ Average OCR similarity: 1.00
predicted : 160 00 40 00 250 0 لننرن علرانى ارع
label : 40.00 160.00 فرخ حلوانى لنشون 0.250

✅ Average OCR similarity: 0.20
predicted : ٥٥ و 122 ,00 MasterCard
label : MasterCard 122.00

✅ Average OCR similarity: 0.00
predicted : GROSS: 368
label : GROSS: 368

✅ Average OCR similarity: 1.00
predicted : ٥٤٢٨ ٥١٤٥
label : RNED DETA

✅ Average OCR similarity: 0.11
predicted : البرن هام المرتجع يوم تلاثذية الطازجة ر ٤ ١روما لبائى ااصحا
label : الأصناف لباقى يوما ١٤ و الطازجة للأغذية يوم/ للمرتجع هام البون

✅ Average OCR similarity: 0.14
predicted : 875.00 875.00
label : 875.00 الصافي 875.00 الباقي

✅ Average OCR similarity: 0.54
predicted : bratui9
label : Gratuity :- --------------

✅ Average OCR similarity: 0.71
predicted : ٨،  ٥ ٥ المستلم
label : المستلم GARDEN

✅ Average OCR similarity: 0.07
predicted : تسجبل اضرببر 631- 296

## ensure images type

In [ ]:
import os

folder_path = '/content/drive/MyDrive/DEPI_Project_Dataset/MIX_Data_Sample/img'  # Replace with your folder path
all_files = os.listdir(folder_path)

file_types = set()
for file in all_files:
    _, extension = os.path.splitext(file)
    if extension:
        file_types.add(extension.lower())

print("File types in the folder:")
for type in file_types:
    print(type)

File types in the folder:
.jpg


In [ ]:
del type


## CSV file

In [ ]:
import pandas as pd

df=pd.DataFrame(data_dict,columns=data_dict.keys())
df.head()

,image path,labled,predicted_ezOCR,similarities_ezOCR
0,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,SALE,SALE,1.0
1,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,Invoice Date: 30-10-2021,Invoice Date: 30-10-2021,1.0
2,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,40.00 160.00 فرخ حلوانى لنشون 0.250,160 00 40 00 250 0 لننرن علرانى ارع,0.2
3,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,MasterCard 122.00,"٥٥ و 122 ,00 MasterCard",0.0
4,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,GROSS: 368,GROSS: 368,1.0


In [ ]:
df.to_csv('/content/drive/MyDrive/DEPI_Project_Dataset/MIX_Data_Sample/EZ_OCR.csv')

# ***tesseract***


In [ ]:
!sudo apt install tesseract-ocr libtesseract-dev tesseract-ocr-ara tesseract-ocr-eng
!pip install pytesseract opencv-python easyocr arabic-reshaper python-bidi matplotlib

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
The following NEW packages will be installed:
  libarchive-dev libleptonica-dev libtesseract-dev tesseract-ocr-ara
0 upgraded, 4 newly installed, 0 to remove and 34 not upgraded.
Need to get 4,388 kB of archives.
After this operation, 17.4 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libarchive-dev amd64 3.6.0-1ubuntu1.4 [581 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libleptonica-dev amd64 1.82.0-3build1 [1,562 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libtesseract-dev amd64 4.1.1-2.1build1 [1,600 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-ara all 1:4.00~git30-7274cfa-1.1 

In [ ]:
# Step 2: Import libraries
import pytesseract

import cv2
import numpy as np
from PIL import Image
from google.colab.patches import cv2_imshow
from arabic_reshaper import ArabicReshaper
from bidi.algorithm import get_display
from matplotlib import pyplot as plt
import os
import json
from glob import glob
from difflib import SequenceMatcher


In [ ]:
# Step 3: Configure Tesseract (Colab specific)
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

In [ ]:
!sudo apt install tesseract-ocr-ara


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr-ara is already the newest version (1:4.00~git30-7274cfa-1.1).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [ ]:
def tesseract_prediction(img_path):
    # Path to your image
    image_path = img_path # Replace this with your image path

    # Load image
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Set language (eng+ara for English and Arabic)
    custom_config = r'-l eng+ara --oem 3 --psm 6'

    # OCR prediction
    text = pytesseract.image_to_string(image_rgb, config=custom_config)

    # Also print full text output
    print("Extracted Text:\n", text)
    return text


In [ ]:
base_path='/content/drive/MyDrive/DEPI_Project_Dataset/MIX_Data_Sample'

/content/drive/MyDrive/DEPI_Project_Dataset/MIX_Data_Sample/img/02da316f-9ad1-45d8-98ce-90ec9816d1ae_line_110.jpg

In [ ]:
def get_label_tesseract(image_path):
    try:
        ann_path = image_path.replace('.jpg','.txt').replace('img','ann')
        with open(ann_path, 'r') as f:
            lines = f.readline()
            lines=eval(lines)
            result = ' '.join(lines)
            return result
    except FileNotFoundError:
        print(f"File not found: {ann_path}")

In [ ]:
# Helper function for text similarity
def average_similarity_tesseract(ground_truths,perdicted):

    def string_similarity(a, b):
        return SequenceMatcher(None, a, b).ratio()

    # Calculate average similarity
    if perdicted:
        similarities = [string_similarity(pred, gt) for pred, gt in zip(perdicted, ground_truths)]
        average_similarity = sum(similarities) / len(similarities)
        print(f"\n✅ Average OCR similarity: {average_similarity:.2f}")
        return average_similarity
    else:
        print("\n⚠️ No valid predictions made.")
        return -1.0


In [ ]:
predictions = []
data_dict={'image path':[] , 'labled':[],'predicted_tesseract' :[],'similarities_tesseract':[]}

for img in glob(os.path.join(base_path, 'img', '*')):

    results_text=tesseract_prediction(img)
    data_dict['image path'].append(img)
    data_dict['predicted_tesseract'].append(results_text)
    label=get_label_tesseract(img)
    data_dict['labled'].append(label)
    data_dict['similarities_tesseract'].append(average_similarity_tesseract(label,results_text))
    predictions.append(results_text)

Extracted Text:
 SALE


✅ Average OCR similarity: 1.00
Extracted Text:
 Invoice Date: 30-10-2021


✅ Average OCR similarity: 1.00
Extracted Text:
 0 لئنشون حلوائي فرع ;469.90 40.00


✅ Average OCR similarity: 0.00
Extracted Text:
 eee aan Oe ite
MasterCard 122.00


✅ Average OCR similarity: 0.06
Extracted Text:
 GROSS: 368


✅ Average OCR similarity: 1.00
Extracted Text:
 RED DETA


✅ Average OCR similarity: 0.11
Extracted Text:
 البون هام للمرتجع/يوع للأغذية الطازجة و 4 ١روما‏ لباقى الأصئاف


✅ Average OCR similarity: 0.08
Extracted Text:
 875.00  يفاصلا‎ eee, Meee
Oe 727 07


✅ Average OCR similarity: 0.33
Extracted Text:
 GF SCULG Seats


✅ Average OCR similarity: 0.12
Extracted Text:
 ‎GARDE‏ المستلم


✅ Average OCR similarity: 0.07
Extracted Text:
 تسجيل ضريبي / 298 - 631 - 220


✅ Average OCR similarity: 0.14
Extracted Text:
 P<EG YMANKARIOUS<<GESKA<OSAMA<MATTA<ATALLA<<<


✅ Average OCR similarity: 0.20
Extracted Text:
 ملبن/نوغا - انواع مقي ‎rr rey‏


✅ Average OCR 

In [ ]:
import pandas as pd

df=pd.DataFrame(data_dict,columns=data_dict.keys())
df.head()

,image path,labled,predicted_tesseract,similarities_tesseract
0,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,SALE,SALE\n,1.000000
1,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,Invoice Date: 30-10-2021,Invoice Date: 30-10-2021\n,1.000000
2,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,40.00 160.00 فرخ حلوانى لنشون 0.250,0 لئنشون حلوائي فرع ;469.90 40.00\n,0.000000
3,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,MasterCard 122.00,eee aan Oe ite\nMasterCard 122.00\n,0.058824
4,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,GROSS: 368,GROSS: 368\n,1.000000


In [ ]:
df.to_csv('/content/drive/MyDrive/DEPI_Project_Dataset/MIX_Data_Sample/tesseract_OCR.csv')

# ***paddle***

In [ ]:
!pip install paddlepaddle paddleocr




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.8/297.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.6/969.6 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 105.6 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=171547fab2a264999ad0b69d7357e6d89dcf83cb8b7ee972577f4c3e9f5ffb03
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire
  Attempting uninstall: opt_einsum
    Found 

In [ ]:
!wget https://github.com/google/fonts/raw/main/ofl/amiri/Amiri-Regular.ttf -O /root/.paddleocr/arabic-font.ttf


/root/.paddleocr/arabic-font.ttf: No such file or directory


In [ ]:
from paddleocr import PaddleOCR
import re
from collections import defaultdict

import cv2
import numpy as np
from PIL import Image
from google.colab.patches import cv2_imshow
from matplotlib import pyplot as plt
import os
import json
from glob import glob
from difflib import SequenceMatcher

## predict

In [ ]:
base_path='/content/drive/MyDrive/DEPI_Project_Dataset/MIX_Data_Sample'

In [ ]:
from paddleocr import PaddleOCR
import re
from collections import defaultdict

def paddel_predict(image_path):
    # Initialize PaddleOCR with Arabic support
    ocr = PaddleOCR(use_angle_cls=True, lang='ar')

    # Perform OCR
    result = ocr.ocr(image_path, cls=False)

    # ✅ Handle case when OCR fails or returns nothing
    if not result or not result[0]:
        print(f"[WARNING] No text detected in image: {image_path}")
        return ''

    # Function to check if text is Arabic
    def is_arabic(text):
        return bool(re.search(r'[\u0600-\u06FF]', text))

    # Group text into lines using Y position
    lines_dict = defaultdict(list)
    line_threshold = 15  # pixels to group into same line

    for line in result[0]:
        box = line[0]
        text = line[1][0]
        x_coords = [pt[0] for pt in box]
        y_coords = [pt[1] for pt in box]
        avg_x = sum(x_coords) / 4
        avg_y = sum(y_coords) / 4

        # Assign to a line group based on Y
        line_key = round(avg_y / line_threshold)
        lines_dict[line_key].append({'text': text, 'x': avg_x})

    # Sort lines top-to-bottom
    sorted_lines = sorted(lines_dict.items(), key=lambda x: x[0])

    # Process each line
    final_lines = []
    for _, line_items in sorted_lines:
        # Determine if this line is mostly Arabic
        arabic_count = sum(1 for item in line_items if is_arabic(item['text']))
        is_arabic_line = arabic_count >= len(line_items) / 2

        # Sort by x (LTR or RTL)
        line_items.sort(key=lambda x: -x['x'] if is_arabic_line else x['x'])

        # Reverse Arabic text elements only
        processed_line = [item['text'][::-1] if is_arabic(item['text']) else item['text'] for item in line_items]
        final_lines.append(' '.join(processed_line))

    # Final predicted text (single line output)
    predicted_text = ' '.join(final_lines)
    print(predicted_text)
    return predicted_text



In [ ]:
def get_label_paddel(image_path):
    try:
        ann_path = image_path.replace('.jpg','.txt').replace('img','ann')
        with open(ann_path, 'r') as f:
            lines = f.readline()
            lines=eval(lines)
            result = ' '.join(lines)
            return result
    except FileNotFoundError:
        print(f"File not found: {ann_path}")

In [ ]:
def average_similarity_paddel(ground_truths,perdicted):

    def string_similarity(a, b):
        return SequenceMatcher(None, a, b).ratio()

    # Calculate average similarity
    if perdicted:
        similarities = [string_similarity(pred, gt) for pred, gt in zip(perdicted, ground_truths)]
        average_similarity = sum(similarities) / len(similarities)
        print(f"\n✅ Average OCR similarity: {average_similarity:.2f}")
        return average_similarity
    else:
        print("\n⚠️ No valid predictions made.")
        return -1.0

In [ ]:
predictions = []
data_dict={'image path':[] , 'labled':[],'predicted_tesseract' :[],'similarities_tesseract':[]}

for img in glob(os.path.join(base_path, 'img', '*')):

    results_text=paddel_predict(img)
    data_dict['image path'].append(img)
    data_dict['predicted_tesseract'].append(results_text)
    label=get_label_paddel(img)
    data_dict['labled'].append(label)
    data_dict['similarities_tesseract'].append(average_similarity_paddel(label,results_text))
    predictions.append(results_text)

[2025/05/01 14:37:25] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, use_gcu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/root/.paddleocr/whl/det/ml/Multilingual_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/root/.paddleocr/whl/rec/arabic/arabic_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch

## csv file

In [ ]:
import pandas as pd

df=pd.DataFrame(data_dict,columns=data_dict.keys())
df.head()

,image path,labled,predicted_tesseract,similarities_tesseract
0,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,SALE,SALE,1.000000
1,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,Invoice Date: 30-10-2021,Invoice Date: 30-10-2021,1.000000
2,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,40.00 160.00 فرخ حلوانى لنشون 0.250,0.250 لنشون حلوائيفرغ 6٥ 0 40,0.068966
3,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,MasterCard 122.00,MasterCard 22,0.923077
4,/content/drive/MyDrive/DEPI_Project_Dataset/MI...,GROSS: 368,GROSS: 368,1.000000


In [ ]:
df.to_csv('/content/drive/MyDrive/DEPI_Project_Dataset/MIX_Data_Sample/paddle_OCR.csv')